### 🚀 Day 20 — ML Pipelines, ColumnTransformer & Data Leakage Prevention
Day 19 completed
Day 20 started
Week 3

Today we're moving from training individual ML models to building professional, reusable machine learning workflows.

Your main project context remains our Credit Card Fraud Detection dataset.
Today's learning objectives
By the end of Day 20, you'll understand:
1. Why preprocessing is necessary.
2. What an ML Pipeline is.
3. How ColumnTransformer works.
4. How to handle numerical and categorical columns.
5. What data leakage is and how to prevent it.
6. How to combine preprocessing and model training into one workflow.

In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report 


In [2]:
# Step 2 — Create a sample dataset
data = {
    "Age":[22,25,47,52,46,56,23,40,35,60],
    "Salary": [25000,30000,60000,80000,55000,
               90000,28000,50000,45000,100000
        ],
        "Purchased": [0,0,1,1,1,1,0,1,0,1] 
}

df = pd.DataFrame(data)
print(df)

   Age  Salary  Purchased
0   22   25000          0
1   25   30000          0
2   47   60000          1
3   52   80000          1
4   46   55000          1
5   56   90000          1
6   23   28000          0
7   40   50000          1
8   35   45000          0
9   60  100000          1


In [3]:
# Our objective: predict whether a customer purchases a product.
# Step 3 — Separate X and y
X = df[["Age", "Salary"]]
y = df["Purchased"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

# Step 4 — Create the Pipeline
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression())
])

# Understand the three steps:
# Step	Purpose
# imputer	Fills missing numerical values
# scaler	Standardizes numerical features
# model	Trains Logistic Regression

# Step 5 — Train and evaluate
pipeline.fit(X_train,y_train)
predictions = pipeline.predict(X_test)

print(classification_report(
    y_test,
    predictions,
    zero_division=0
))

# That's your first complete ML Pipeline!

# Important: This tiny dataset is only for learning the workflow. Its test metrics aren't reliable evidence of real-world performance.

              precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3



In [5]:
# 3. Data Leakage — one of the most important ML concepts
# Before learning ColumnTransformer, understand this problem.

# Data leakage happens when information from outside the training data improperly influences model training.

# For example, suppose you have 10,000 records:
# - 8,000 training records
# - 2,000 test records
# You calculate the average salary using all 10,000 records, fill missing values, and then split the data.

#Split First
X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

# Fit preprocessing only on the training data
scaler = StandardScaler()
X_trained_scaled = scaler.fit_transform(X_train)

# Only transform test data
X_test_scaled = scaler.transform(X_test)

# Remember these two methods:
# Method	Meaning
# fit()	Learn parameters from data
# transform()	Apply learned parameters to data
# fit_transform()	Learn and apply

# A Pipeline handles this correctly when you call fit() on training data. During prediction, it only applies the already-learned transformations.

In [ ]:
# 4. ColumnTransformer — handle different column types
# Real-world datasets often contain different types of features.

# Example:
# Age	Salary	City	Education
# 25	30000	Delhi	Graduate
# 35	60000	Mumbai	Postgraduate
# 42	80000	Pune	Graduate

# Problem:
# - Age and Salary are numerical.
# - City and Education are categorical.
# - Numerical features need numerical preprocessing.
# - Categorical features need encoding.
# This is where ColumnTransformer helps.

In [6]:
data = {
    "Age": [22, 25, 47, 52, 46, 56, 23, 40, 35, 60,
            28, 32, 45, 50, 29, 38, 42, 55, 31, 48],
    "Salary": [25000, 30000, 60000, 80000, 55000,
               90000, 28000, 50000, 45000, 100000,
               32000, 40000, 70000, 85000, 35000,
               52000, 65000, 95000, 42000, 75000],
    "City": ["Delhi", "Mumbai", "Pune", "Delhi", "Pune",
             "Mumbai", "Delhi", "Pune", "Mumbai", "Delhi",
             "Pune", "Mumbai", "Delhi", "Pune", "Mumbai",
             "Delhi", "Pune", "Mumbai", "Delhi", "Pune"],
    "Education": ["Graduate", "Postgraduate", "Graduate",
                  "Postgraduate", "Graduate", "Postgraduate",
                  "Graduate", "Graduate", "Postgraduate",
                  "Postgraduate", "Graduate", "Graduate",
                  "Postgraduate", "Graduate", "Graduate",
                  "Postgraduate", "Graduate", "Postgraduate",
                  "Graduate", "Postgraduate"],
    "Purchased": [0, 0, 1, 1, 1, 1, 0, 1, 0, 1,
                  0, 0, 1, 1, 0, 1, 1, 1, 0, 1]
}

df = pd.DataFrame(data)

X = df.drop("Purchased", axis=1)
y = df["Purchased"]


# Step 2 — Identify numerical and categorical columns
numeric_features = ["Age", "Salary"]
categorical_features = ["City", "Education"]

In [7]:
# Step 3 — Create preprocessing pipelines
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# What's OneHotEncoder?

# It converts categories into numerical columns.

# For example:
# City	Delhi	Mumbai	Pune
# Delhi	1	0	0
# Mumbai	0	1	0
# Pune	0	0	1

# Step 4 — Combine using ColumnTransformer
preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

# This tells Scikit-learn:
# - Apply numerical preprocessing to Age and Salary.
# - Apply categorical preprocessing to City and Education.
# - Combine the resulting features.

# Step 5 — Create the complete ML Pipeline
full_pipeline = Pipeline([
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

# Step 6 — Train and evaluate
X_train, X_test, y_train, y_test = train_test_split(
    X,y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

full_pipeline.fit(X_train,y_train)
predictions = full_pipeline.predict(X_test)
print(classification_report(
     y_test,predictions,zero_division=0
))

# You have now built a complete preprocessing and model-training workflow.

              precision    recall  f1-score   support

           0       1.00      1.00      1.00         2
           1       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



In [9]:
# 5. Apply today's learning to your Fraud Detection project
# Your Credit Card Fraud Detection dataset contains numerical features (V1–V28, Time, Amount).

# Therefore, you don't need categorical encoding for this dataset.

# Fit only on training data 

# You can create a clean pipeline like this:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

fraud_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    ))
])

# Keep only numeric columns for your fraud pipeline
X_train_numeric = X_train.select_dtypes(include=['int64', 'float64'])
X_test_numeric = X_test.select_dtypes(include=['int64', 'float64'])

# Then fit your pipeline on the numeric data
fraud_pipeline.fit(X_train_numeric, y_train)
fraud_predictions = fraud_pipeline.predict(X_test_numeric)

print("Confusion Matrix:")
print(confusion_matrix(y_test, fraud_predictions))

print("\nClassification Report:")
print(classification_report(
    y_test,
    fraud_predictions,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

Confusion Matrix:
[[2 0]
 [0 3]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00         2
       Fraud       1.00      1.00      1.00         3

    accuracy                           1.00         5
   macro avg       1.00      1.00      1.00         5
weighted avg       1.00      1.00      1.00         5



Day 25 — PyTorch Fundamentals
Deep Learning Begins
 Day 25 of 30

1. What is PyTorch?
PyTorch is a Python framework used to build and train machine learning and deep learning models.

Think about the difference:
Traditional ML	Deep Learning
Logistic Regression	Neural Networks
Decision Tree	Deep Neural Networks
Random Forest	CNNs, RNNs, Transformers
Scikit-learn	PyTorch

Why learn PyTorch?
- Build neural networks.
- Train deep learning models.
- Work with GPUs.
- Build computer vision applications.
- Work with NLP and modern AI models.

### Day 25 — PyTorch Fundamentals
Deep Learning Begins
 Day 25 of 30

1. What is PyTorch?
PyTorch is a Python framework used to build and train machine learning and deep learning models.

Think about the difference:
Traditional ML	Deep Learning
Logistic Regression	Neural Networks
Decision Tree	Deep Neural Networks
Random Forest	CNNs, RNNs, Transformers
Scikit-learn	PyTorch

Why learn PyTorch?
- Build neural networks.
- Train deep learning models.
- Work with GPUs.
- Build computer vision applications.
- Work with NLP and modern AI models.

3. Understand Tensors
Today's most important concept
A tensor is a data structure used to store numerical values in PyTorch.

You can think of tensors as an extension of NumPy arrays.
Type	Example	Dimension
Scalar	10	0D
Vector	[10, 20, 30]	1D
Matrix	[[1, 2], [3, 4]]	2D
3D Tensor	Multiple matrices	3D

In [ ]:

import torch

# 1. Scalar (0D Tensor)
scalar = torch.tensor(10)

# 2. Vector (1D Tensor)
vector = torch.tensor([10, 20, 30])

# 3. Matrix (2D Tensor)
matrix = torch.tensor([
    [1, 2],
    [3, 4]
])

# 4. 3D Tensor
tensor_3d = torch.tensor([
    [[1, 2], [3, 4]],
    [[5, 6], [7, 8]]
])

print("Scalar:", scalar)
print("Vector:", vector)
print("Matrix:\n", matrix)
print("3D Tensor:\n", tensor_3d)

# Check dimensions
print("\nDimensions:")
print(scalar.ndim)
print(vector.ndim)
print(matrix.ndim)
print(tensor_3d.ndim)

# Check shapes
print("\nShapes:")
print(scalar.shape)
print(vector.shape)
print(matrix.shape)
print(tensor_3d.shape)

# Understand these two properties:
# - ndim: Number of dimensions.
# - shape: Size of the tensor along each dimension.
# For example, a matrix with 2 rows and 3 columns has shape (2, 3).

Scalar: tensor(10)
Vector: tensor([10, 20, 30])
Matrix:
 tensor([[1, 2],
        [3, 4]])
3D Tensor:
 tensor([[[1, 2],
         [3, 4]],

        [[5, 6],
         [7, 8]]])

Dimensions:
0
1
2
3

Shapes:
torch.Size([])
torch.Size([3])
torch.Size([2, 2])
torch.Size([2, 2, 2])


In [2]:
# 4. Tensor Operations
# PyTorch allows you to perform mathematical operations directly on tensors.

# Add the following code below your previous exercise:

import torch

a = torch.tensor([10, 20, 30])
b = torch.tensor([1, 2, 3])

# Addition
print("Addition:", a + b)

# Subtraction
print("Subtraction:", a - b)

# Multiplication (element-wise)
print("Multiplication:", a * b)

# Division
print("Division:", a / b)

# Matrix multiplication
x = torch.tensor([
    [1, 2],
    [3, 4]
])

y = torch.tensor([
    [5, 6],
    [7, 8]
])

print("Matrix Multiplication:\n", torch.matmul(x, y))

# Other useful operations
print("Mean:", a.float().mean())
print("Maximum:", a.max())
print("Minimum:", a.min())




Addition: tensor([11, 22, 33])
Subtraction: tensor([ 9, 18, 27])
Multiplication: tensor([10, 40, 90])
Division: tensor([10., 10., 10.])
Matrix Multiplication:
 tensor([[19, 22],
        [43, 50]])
Mean: tensor(20.)
Maximum: tensor(30)
Minimum: tensor(10)


Expected addition:
tensor([11, 22, 33])


One important distinction:
- a * b performs element-wise multiplication.
- torch.matmul(a, b) performs matrix multiplication when the tensor shapes are compatible.
These operations are fundamental to neural network calculations.

5. CPU vs GPU
When training deep learning models, we often perform millions or billions of mathematical operations.

A CPU can handle these calculations, but GPUs are designed to perform many operations in parallel.

In [3]:

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("GPU Available!")
else:
    device = torch.device("cpu")
    print("Using CPU!")

numbers = torch.tensor(
    [10, 20, 30],
    device=device
)

print("Tensor:", numbers)
print("Device:", numbers.device)


Using CPU!
Tensor: tensor([10, 20, 30])
Device: cpu


6. Autograd — Automatic Differentiation
Extremely important for Deep Learning
Before learning neural networks, understand one important concept: Gradient.

Imagine your model predicts a student's marks:
- Actual marks = 80
- Predicted marks = 60
Your model has made an error.

During training, it needs to adjust its parameters to reduce that error.

This is where gradients become important.

PyTorch provides a feature called autograd, which automatically calculates gradients.

In [4]:

import torch

# Create a tensor that tracks gradients
x = torch.tensor(3.0, requires_grad=True)

# Mathematical operation
y = x ** 2

# Calculate gradient
y.backward()

print("Value of x:", x.item())
print("Value of y:", y.item())
print("Gradient:", x.grad.item())


Value of x: 3.0
Value of y: 9.0
Gradient: 6.0


What happened?
1. We created x with requires_grad=True.
2. PyTorch tracked the mathematical operation.
3. y.backward() calculated the derivative.
4. x.grad returned the gradient.
Why is this useful?

Neural networks have many parameters. Instead of calculating every derivative manually, PyTorch calculates the gradients automatically.

7. Your first mini training workflow
Now let's connect the concepts.

We'll train a very simple linear model using PyTorch.

Our goal:

Given some input values, predict their corresponding output values.

We'll use this relationship:

y = 2x + 1

Our model will try to learn the relationship from data.

In [ ]:

import torch
import torch.nn as nn

# 1. Create training data
X = torch.tensor(
    [[1.0], [2.0], [3.0], [4.0], [5.0]]
)

y = torch.tensor(
    [[3.0], [5.0], [7.0], [9.0], [11.0]]
)

# 2. Create a simple linear model
model = nn.Linear(1, 1)

# 3. Define loss function
loss_function = nn.MSELoss()

# 4. Define optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

# 5. Training loop
for epoch in range(1000):

    # Forward pass
    predictions = model(X)

    # Calculate error
    loss = loss_function(predictions, y)

    # Clear previous gradients
    optimizer.zero_grad()

    # Calculate new gradients
    loss.backward()

    # Update model parameters
    optimizer.step()

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch+1}, Loss: {loss.item():.4f}"
        )

# 6. Test the trained model
test_input = torch.tensor([[6.0]])

with torch.no_grad():
    prediction = model(test_input)

print("\nPrediction for input 6:")
print(prediction.item())

print("\nLearned weight:")
print(model.weight.item())

print("Learned bias:")
print(model.bias.item())


Your model should learn a weight close to 2 and a bias close to 1.

For input 6, its prediction should approach 13.

The exact values may vary slightly because the model starts with randomly initialized parameters.
8. Understand the training workflow
This is the foundation of PyTorch training:
1. Forward pass
   
   Model receives input and generates predictions.
2. Loss calculation
   
   Compare predictions against actual values.
3. Zero gradients
   
   Clear gradients from the previous training iteration.
4. Backpropagation
   
   loss.backward() calculates gradients.
5. Optimizer step
   
   optimizer.step() updates the model parameters.
6. Repeat
   
   Continue training over multiple epochs.
You'll use this same fundamental workflow when building neural networks on Day 26.

#### Summary Day-25

Today's concepts:
Topic	What you learned
PyTorch	Deep learning framework
Tensors	Data representation
Tensor operations	Mathematical calculations
CPU vs GPU	Processing devices
Autograd	Automatic gradient calculation
Forward pass	Generate predictions
Loss function	Measure prediction error
Backpropagation	Calculate gradients
Optimizer	Update model parameters

### What's next — Day 26
Next lesson
Artificial Neural Networks (ANN)

Day 26 — Artificial Neural Networks (ANN)
Deep Learning — Practical Implementation
 Day 26 of 30

Today's project: Build a neural network that predicts whether a breast tumor is benign or malignant using the Breast Cancer Wisconsin dataset.

By the end, you'll understand how neural networks learn from data and how to train and evaluate one using PyTorch.
1. What is an Artificial Neural Network?
An Artificial Neural Network is a machine learning model inspired by the way biological neurons process information.

In traditional ML, you used algorithms such as:
- Logistic Regression
- Decision Tree
- Random Forest
Now we'll use layers of interconnected artificial neurons to learn patterns from data.

We'll build a proper neural network and learn:
- Input, hidden, and output layers.
- Weights and biases.
- Activation functions: ReLU and Sigmoid.
- Loss functions and optimizers.
- Backpropagation.
- Training an ANN on a real dataset.

Your focus today: Understand tensors, autograd, and the training loop. These are the building blocks for everything we'll do in deep learning from Day 26 onward.

An ANN generally consists of three types of layers:
Layer	Purpose
Input layer	Receives input features
Hidden layers	Learn patterns from data
Output layer	Produces the final prediction

For our project:
- Input layer: 30 features.
- Hidden layer 1: 16 neurons.
- Hidden layer 2: 8 neurons.
- Output layer: 1 neuron.

In [5]:
# 5. Build the ANN project
# Step 1 — Project structure  
# Step 2 — Check dependencies
# You should already have PyTorch and scikit-learn installed.
# Step 3 — Complete ANN implementation

# ==========================================================
# DAY 26 - ARTIFICIAL NEURAL NETWORK (ANN)
# Project: Breast Cancer Classification using PyTorch
# ==========================================================

import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

import numpy as np
from pathlib import Path


# ----------------------------------------------------------
# 1. DEVICE CONFIGURATION
# ----------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# ----------------------------------------------------------
# 2. LOAD DATASET
# ----------------------------------------------------------

dataset = load_breast_cancer()

X = dataset.data
y = dataset.target

print("\nDataset Shape:", X.shape)
print("Target Shape:", y.shape)

print("Class Names:", dataset.target_names)


# ----------------------------------------------------------
# 3. TRAIN-TEST SPLIT
# ----------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ----------------------------------------------------------
# 4. FEATURE SCALING
# ----------------------------------------------------------

scaler = StandardScaler()

# Fit scaler only on training data
X_train = scaler.fit_transform(X_train)

# Apply training scaler to test data
X_test = scaler.transform(X_test)


# ----------------------------------------------------------
# 5. CONVERT DATA INTO PYTORCH TENSORS
# ----------------------------------------------------------

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32
).reshape(-1, 1)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32
).reshape(-1, 1)


# ----------------------------------------------------------
# 6. CREATE DATA LOADER
# ----------------------------------------------------------

train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)


# ----------------------------------------------------------
# 7. BUILD NEURAL NETWORK
# ----------------------------------------------------------

class BreastCancerANN(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(30, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)

        )

    def forward(self, x):

        return self.network(x)


model = BreastCancerANN().to(device)


# ----------------------------------------------------------
# 8. LOSS FUNCTION AND OPTIMIZER
# ----------------------------------------------------------

loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)


# ----------------------------------------------------------
# 9. TRAINING LOOP
# ----------------------------------------------------------

epochs = 50

print("\nStarting ANN Training...\n")

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for batch_X, batch_y in train_loader:

        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)

        # Forward pass
        predictions = model(batch_X)

        # Calculate loss
        loss = loss_function(
            predictions,
            batch_y
        )

        # Clear previous gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update parameters
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    if (epoch + 1) % 5 == 0:

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {average_loss:.4f}"
        )


# ----------------------------------------------------------
# 10. MODEL EVALUATION
# ----------------------------------------------------------

model.eval()

with torch.no_grad():

    X_test_tensor = X_test_tensor.to(device)

    logits = model(X_test_tensor)

    probabilities = torch.sigmoid(logits)

    predictions = (
        probabilities >= 0.5
    ).int()

    y_pred = predictions.cpu().numpy().flatten()


# ----------------------------------------------------------
# 11. EVALUATION REPORT
# ----------------------------------------------------------

accuracy = accuracy_score(y_test, y_pred)

print("\n================================")
print("ANN EVALUATION RESULTS")
print("================================")

print("Accuracy:", round(accuracy, 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=dataset.target_names
    )
)


# ----------------------------------------------------------
# 12. SAMPLE PREDICTION
# ----------------------------------------------------------

sample = X_test_tensor[:1]

with torch.no_grad():

    sample_logit = model(sample)

    sample_probability = torch.sigmoid(
        sample_logit
    ).item()

    sample_prediction = (
        "Malignant"
        if sample_probability >= 0.5
        else "Benign"
    )

print("\nSample Prediction:")
print("Probability of Malignant:", round(sample_probability, 4))
print("Predicted Class:", sample_prediction)


# ----------------------------------------------------------
# 13. SAVE MODEL
# ----------------------------------------------------------

project_root = Path(__file__).resolve().parent.parent

models_dir = project_root / "models"
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "breast_cancer_ann.pth"

torch.save(
    model.state_dict(),
    model_path
)

print("\nANN Model Saved Successfully!")
print("Location:", model_path)

print("\nDAY 26 ANN PROJECT COMPLETED!")


Using device: cpu

Dataset Shape: (569, 30)
Target Shape: (569,)
Class Names: ['malignant' 'benign']

Starting ANN Training...

Epoch [5/50] Loss: 0.4356
Epoch [10/50] Loss: 0.1600
Epoch [15/50] Loss: 0.0897
Epoch [20/50] Loss: 0.0729
Epoch [25/50] Loss: 0.0780
Epoch [30/50] Loss: 0.0491
Epoch [35/50] Loss: 0.0449
Epoch [40/50] Loss: 0.0410
Epoch [45/50] Loss: 0.0374
Epoch [50/50] Loss: 0.0352

ANN EVALUATION RESULTS
Accuracy: 0.9561

Confusion Matrix:
[[41  1]
 [ 4 68]]

Classification Report:
              precision    recall  f1-score   support

   malignant       0.91      0.98      0.94        42
      benign       0.99      0.94      0.96        72

    accuracy                           0.96       114
   macro avg       0.95      0.96      0.95       114
weighted avg       0.96      0.96      0.96       114


Sample Prediction:
Probability of Malignant: 0.0
Predicted Class: Benign


NameError: name '__file__' is not defined

You should see:
- Dataset dimensions.
- Training progress every five epochs.
- Decreasing loss in a typical successful run.
- Accuracy and confusion matrix.
- Classification report.
- Sample prediction.
- Saved ANN model.
Your exact accuracy will vary slightly depending on the training process and environment. Don't assume a particular result until you run it.

6. Understand the important training concepts
These are the most important parts of today's code.
Code	Meaning
nn.Linear(30, 16)	Connects 30 inputs to 16 neurons
nn.ReLU()	Adds nonlinearity
nn.Sequential()	Combines layers into one model
nn.BCEWithLogitsLoss()	Measures binary classification error
torch.optim.Adam()	Optimizes model weights
loss.backward()	Calculates gradients
optimizer.step()	Updates parameters
model.eval()	Switches to evaluation mode
torch.no_grad()	Disables gradient tracking during inference
torch.sigmoid()	Converts logits to probabilities

What is an epoch?
An epoch means the model has processed the entire training dataset once.

Instead of processing the entire dataset at once, the model processes 32 samples per batch.

This makes training more manageable, especially for larger datasets.
What is Adam?
Adam is an optimization algorithm that adjusts the model's weights using the calculated gradients.

You can think of it as the mechanism that helps the neural network improve its predictions.

##### Summary Day-26

7. What you accomplished today
Day 26 learning objectives
- ANN architecture: input, hidden, and output layers.
- Weights and biases.
- ReLU and sigmoid.
- Binary classification using neural networks.
- BCEWithLogitsLoss.
- Adam optimizer.
- Batch training and epochs.
- Training and evaluation using PyTorch.
- Saving a trained neural network.
Your project now contains two different model types:
1. Traditional ML: Credit Card Fraud Detection using Logistic Regression.
2. Deep Learning: Breast Cancer Classification using PyTorch ANN.

### Day 27 — NLP & Hugging Face Transformers
Deep Learning — NLP
 Day 27 of 30


Today's project: Build an AI-powered Sentiment Analysis application that understands whether a sentence expresses positive or negative sentiment.
1. What is NLP?
NLP (Natural Language Processing) is a field of AI that enables computers to understand and process human language.

Examples of NLP applications:
- ChatGPT and AI assistants.
- Text summarization.
- Sentiment analysis.
- Language translation.
- Spam detection.
- Question answering.
- Text classification.

2. What is Tokenization?
Computers don't directly understand sentences like humans do.

Before processing text, NLP models convert text into smaller units called tokens.

Example:
I love Machine Learning


A tokenizer might split it into:
["I", "love", "Machine", "Learning"]


The tokenizer then converts those tokens into numerical token IDs.

Important terminology
Term	Meaning
Token	A piece of text
Tokenizer	Converts text into model-compatible tokens
Token ID	Numerical representation of a token
Attention mask	Indicates which tokens are real input versus padding
Embedding	Numerical vector representing a token

3. What are Transformers?
Transformer architecture is an important neural network architecture used in modern AI.

Traditional NLP approaches often struggled with understanding relationships between words across long sentences.

Transformers use a mechanism called self-attention to learn relationships between different tokens in a sequence.

For example:
The animal didn't cross the road because it was tired.

A language model needs to understand what "it" refers to from the surrounding context.

Self-attention helps models learn these contextual relationships.

You don't need to implement a Transformer from scratch today. We'll use an already-trained model.
4. What is Hugging Face?
Hugging Face provides tools, models, datasets, and libraries for machine learning and NLP.

Its popular library is:

Transformers

It gives developers access to pretrained models without having to train everything from scratch.

Examples of pretrained models:
- BERT
- DistilBERT
- RoBERTa
- GPT
- T5

Official resources:
- Hugging Face Model Hub
- Transformers Documentation
5. Build your first NLP application
Main Day 27 practical project
We'll create a sentiment analyzer using a pretrained DistilBERT model.

Model:

distilbert/distilbert-base-uncased-finetuned-sst-2-english

This model is already trained to classify English text as positive or negative.

In [ ]:
# SENTIMENT

# ==========================================================
# DAY 27 - NLP WITH HUGGING FACE TRANSFORMERS
# Project: Sentiment Analysis Application
# ==========================================================

from transformers import pipeline, AutoTokenizer

# ----------------------------------------------------------
# 1. SELECT PRETRAINED MODEL
# ----------------------------------------------------------

MODEL_NAME = (
    "distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

print("Loading pretrained NLP model...")

# ----------------------------------------------------------
# 2. LOAD TOKENIZER
# ----------------------------------------------------------

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ----------------------------------------------------------
# 3. LOAD SENTIMENT ANALYSIS PIPELINE
# ----------------------------------------------------------

sentiment_analyzer = pipeline(
    "sentiment-analysis",
    model=MODEL_NAME
)

print("Model loaded successfully!")


# ----------------------------------------------------------
# 4. USER INPUT
# ----------------------------------------------------------

print("\n================================")
print("AI SENTIMENT ANALYZER")
print("================================")

while True:

    text = input(
        "\nEnter a sentence (or type exit): "
    ).strip()

    if text.lower() == "exit":
        print("Application closed.")
        break

    if not text:
        print("Please enter some text.")
        continue

    # ------------------------------------------------------
    # 5. TOKENIZATION
    # ------------------------------------------------------

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=512
    )

    print("\nToken IDs:")
    print(tokens["input_ids"])

    # ------------------------------------------------------
    # 6. SENTIMENT PREDICTION
    # ------------------------------------------------------

    result = sentiment_analyzer(text)

    label = result[0]["label"]
    score = result[0]["score"]

    # ------------------------------------------------------
    # 7. DISPLAY RESULTS
    # ------------------------------------------------------

    print("\nPrediction Results")
    print("------------------")

    print("Sentiment:", label)
    print("Confidence:", round(score * 100, 2), "%")

    print("\nAnalysis completed!")


c:\Users\netware\Desktop\AI_PLAYGROUND\RAJ_PY\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading pretrained NLP model...


Loading weights: 100%|██████████| 104/104 [00:07<00:00, 14.02it/s]


Model loaded successfully!

AI SENTIMENT ANALYZER

Token IDs:
[101, 1045, 2572, 3110, 3407, 999, 102]

Prediction Results
------------------
Sentiment: POSITIVE
Confidence: 99.99 %

Analysis completed!


Try these inputs:
Test sentences
"I absolutely love learning Artificial Intelligence!"Expected: POSITIVE
 Copy

"This application is terrible and completely useless."Expected: NEGATIVE
 Copy

"The product is amazing and works perfectly."Expected: POSITIVE
 Copy

"I am disappointed with the quality."Expected: NEGATIVE
 Copy


Your output will look approximately like:
AI SENTIMENT ANALYZER

Enter a sentence:
I absolutely love learning Artificial Intelligence!

Prediction Results
------------------
Sentiment: POSITIVE
Confidence: 99.XX %

Analysis completed!

The exact confidence score depends on the input and model.


Understand what happened behind the scenes
You only wrote a few lines of model-loading code, but several things happened.
Your NLP application architecture
User enters text


Tokenizer

Text → Token IDs + Attention Mask


Pretrained DistilBERT

Processes contextual information


Sentiment prediction
POSITIVE / NEGATIVE
Classification label and model score


What does confidence mean?
For example:
{
  "label": "POSITIVE",
  "score": 0.998
}


This means the model assigned approximately 99.8% of its classification score to the positive class.

Important: This is the model's confidence score, not a guarantee that the sentence is objectively positive. The model is also designed for English sentiment classification, not every language or every type of text.

##### Summary Day-27
Concept	Status
NLP fundamentals	Covered
Tokenization	Implemented
Token IDs	Implemented
Attention masks	Introduced
Transformers	Explained
Pretrained models	Implemented
Hugging Face	Implemented
Sentiment analysis	Project provided

### 8. Day 28 — Embeddings, Vector Databases & RAG

We'll cover:
1. What are embeddings?
2. Semantic search.
3. Vector databases.
4. Similarity search.
5. Retrieval-Augmented Generation (RAG).
6. Build a document question-answering application.